# C4 · Ajuste de PSF (psffit)

**Spec:** [`docs/spec_C4_codex_psf_fitting.md`](../docs/spec_C4_codex_psf_fitting.md)  |  **Bloque:** C · Extracción  |  **Run de este set:** `ROXs42Bb_realigned`

Ajusta simultáneamente estrella + compañero sobre el modelo de PSF (método canónico).

| | |
|---|---|
| **Entrada** | Cubo + PSF (C1) |
| **Salida (QC/productos)** | `stages/spec_psffit_qc.json` |
| **Consume aguas abajo** | D1, D2, E1, E3 (método canónico) |


## Qué hace C4 y por qué es el canónico

C4 (`psffit`) es el **método primario** recomendado por la literatura para un compañero a ~1.75″. Por canal ajusta un modelo **lineal** por mínimos cuadrados:

```
D(y,x) = a·P_estrella + b·P_compañero + (c₀ + c₁·y + c₂·x)
```

Las incógnitas por canal son solo `(a, b, c₀, c₁, c₂)`: `a` amplitud de la estrella, `b` la del compañero, y `(c₀,c₁,c₂)` un **plano de fondo local**. Toda la no-linealidad (forma de PSF, posiciones) quedó resuelta aguas arriba (C1/B3) → el problema es lineal, exacto y rápido. **Esa es la decisión de diseño central.**

**Por qué es el canónico:** ajusta estrella + compañero + fondo **simultáneamente**, atacando la decontaminación del halo de frente, **sin sustracción agresiva**. El plano `(c₀,c₁,c₂)` absorbe el gradiente del halo → es el método **menos sesgado** (por eso su media de controles es la más pequeña; ver el trabajo de referenciación en C3/D2).

**Calidad del ajuste:** χ²ᵣ ≈ 1.03 (excelente), número de condición 10.7 (bien condicionado), `rho_ab` 0.17 (estrella y compañero **separables**), crosstalk 0.03 (las líneas de la estrella no contaminan al compañero), y la estrella se recupera (flujo psffit / apertura grande = 1.014).

**Salvedad:** `rho_bc` mediana 0.45 (1303 canales >0.5) — el compañero es débil y queda parcialmente degenerado con el plano de fondo. Errores empíricos (M5 rojo). El producto final (`spec_final_object`) lleva además `cont_runmed_biasref` (referenciado a controles, del trabajo de D2) para G3.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_x03_psffit.sh --run-id $RUN
```

Moderado (~3 min con Psfao lru_cache; sin cache era 2h+).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/spec_psffit_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_x03_psffit.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/spec_psffit_qc.json', RUN_ID)
nb.show(qc, keys=['chi2r.median', 'condition_number_median', 'rho_ab_median', 'vs_large_aperture_median_ratio', 'v3_star_scale_ok'], title='C4')


## Los chequeos del QC, en físico

C4 ajusta **dos PSF a la vez** (primaria + compañero) canal a canal, así que sus chequeos preguntan por lo que puede salir mal en ese ajuste conjunto.

| Chequeo | ¿Qué pregunta contesta? | Si falla |
|---|---|---|
| `v3_star_scale_ok` | **¿El ajuste reproduce la estrella que sí vemos bien?** Razón entre el espectro de la primaria ajustada y su fotometría de apertura grande: mediana en [0.97, 1.03] y sin pendiente con λ. | Si el modelo no reproduce la fuente brillante, el residuo donde vive el compañero tampoco es de fiar. |
| `v4_rho_ab_ok` | **¿Son separables las dos fuentes?** Correlación ρ(a,b) entre las amplitudes de primaria y compañero, y número de condición del ajuste; a esta separación se espera \|ρ\| < 0.3. | Con ρ→1 el ajuste no puede decidir cuánta luz es de cada una: el flujo del compañero se vuelve **degenerado** (cualquier reparto encaja igual de bien) y su error real es mucho mayor que el formal. |

Las otras verificaciones de la spec (χ²ᵣ~1, residuo limpio, crosstalk, contraste con C2/C3) se revisan en los plots de abajo, no como banderas del QC.


## Resultados que llevaron a la conclusión

Región de ajuste, condicionamiento, χ²ᵣ, crosstalk y validación de la estrella del `spec_psffit_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('C4', 'stages/spec_psffit_qc.json'):
        q = nb.load_qc('stages/spec_psffit_qc.json', RUN_ID)
        fr = q['fit_region']; cond = q['conditioning']; ck = q['checks']
        print(f"región: estrella r={fr['star_radius_px']:.0f}px, compañero r={fr['comp_radius_px']:.0f}px, {fr['n_pixels_median']:.0f} px/ajuste")
        print(f"χ²ᵣ: mediana {q['chi2r']['median']:.3f} (p90 {q['chi2r']['p90']:.2f})")
        print(f"condicionamiento: cond={cond['condition_number_median']:.1f}, "
              f"rho_ab(estrella-compañero)={cond['rho_ab_median']:.2f}, "
              f"rho_bc(compañero-fondo)={cond['rho_bc_median']:.2f} ({cond['channels_rho_bc_gt_0p5']} canales >0.5)")
        print(f"crosstalk (líneas estrella->compañero) = {q['crosstalk']['metric_corr_b_vs_a_lines']:.3f}")
        print(f"estrella recuperada (psffit/apertura grande) = {q['star_product_check']['vs_large_aperture_median_ratio']:.3f}")
        print(f"checks: v3_star_scale={ck['v3_star_scale_ok']} v4_rho_ab={ck['v4_rho_ab_ok']} rho_bc_warning={ck['rho_bc_warning']}")
        print(f"errores: {q['errors']['mode']} ({q['errors']['n_controls']} controles); open_issue: {q['open_issues'][0]}")


## Plot 1 — el cubo residual: estrella + compañero removidos

Entrada (`stage02`) vs residual del ajuste (`cube_psffit_residual.fits`), colapsados en 7000–8500 Å. El halo estelar casi desaparece (queda solo el residuo del anillo ~4–5% de C1 en el núcleo) y el compañero también → demuestra la decontaminación **simultánea** y el χ²ᵣ≈1. `+` estrella, `○` compañero.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    from musepipe.io import read_wavelength_axis
    rd = nb.run_dir(RUN_ID)
    q = nb.load_qc('stages/spec_psffit_qc.json', RUN_ID)
    loc = nb.load_qc('stages/stage01c_qc.json', RUN_ID)   # posiciones oficiales (B3)
    (py, px), (cy, cx) = loc['primary']['pos_yx'], loc['companion']['pos_yx']
    def cube(path):
        h = fits.open(path); hd = next(x for x in h if x.data is not None)
        d = np.asarray(hd.data, float); h.close(); return d[0] if d.ndim == 4 else d
    res = cube(q['products']['residual_cube'])
    inp = cube(rd / 'stages' / 'stage02_xcorr_cube_stack.fits')
    with fits.open(rd / 'stages' / 'stage02_xcorr_cube_stack.fits') as _h:
        wave = read_wavelength_axis(_h)   # ext WAVELENGTH del stack
    sel = (wave >= 7000) & (wave <= 8500)
    imgi = np.nanmedian(inp[sel], axis=0); imgr = np.nanmedian(res[sel], axis=0)
    v = np.nanpercentile(imgi, [30, 99.5])
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
    for ax, im, t in [(axes[0], imgi, 'entrada: estrella + compañero'),
                      (axes[1], imgr, 'residual psffit: ambos removidos (χ²ᵣ≈1)')]:
        ax.imshow(im, origin='lower', cmap='magma', vmin=v[0], vmax=v[1])
        ax.plot(px, py, '+', color='cyan', ms=10); ax.plot(cx, cy, 'o', mfc='none', mec='lime', ms=12)
        ax.set_title(t); ax.axis('off')
    fig.tight_layout()
    outdir = rd / 'plots' / 'c4_psffit'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'residual.png', dpi=110); print('figura ->', outdir / 'residual.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — el espectro canónico del compañero (no-detección)

`spec_psffit_object.fits` con la banda ±1σ empírica de los controles y Hα. El continuo sube al rojo (SED real de enana fría) y **no hay nada en Hα** — la no-detección con el método canónico. El azul (λ<7000) está dominado por ruido (SNR<1).


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from astropy.io import fits
    rd = nb.run_dir(RUN_ID)
    h = fits.open(rd / 'stages' / 'spec_psffit_object.fits')
    wave = np.asarray(h[1].data['wave_A'], float)   # eje λ del propio producto
    flux = np.asarray(h[1].data['flux'], float); h.close()
    C = np.load(rd / 'stages' / 'spec_psffit_controls.npz')['control_spectra']
    sig = np.nanstd(C, axis=0)
    from musepipe.spectral import median_filter_1d
    sm = median_filter_1d(flux, 41)   # mediana móvil, ignora NaN
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.fill_between(wave, -sig, sig, color='0.85', label=f'±1σ empírico ({C.shape[0]} controles)')
    ax.plot(wave, flux, lw=0.3, color='0.55', alpha=0.6)
    ax.plot(wave, sm, lw=1.3, color='tab:blue', label='flujo compañero psffit (suavizado)')
    ax.axvline(6563, color='tab:red', ls=':', label='Hα'); ax.axhline(0, color='0.6', lw=0.6)
    ax.set_ylim(np.nanpercentile(flux, 2), np.nanpercentile(flux, 98))
    ax.set_xlabel('λ [Å]'); ax.set_ylabel('flujo (canónico)')
    ax.set_title('C4 · espectro canónico psffit del compañero — nada en Hα (no-detección)')
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = rd / 'plots' / 'c4_psffit'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'spectrum.png', dpi=110); print('figura ->', outdir / 'spectrum.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'psffit'
    PRODUCT_P = 'spec_psffit_object.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    _cols = list(_d.columns.names)
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    # El empírico manda; `flux_err` es el que eligió la etapa y solo
    # aporta algo cuando NO es el empírico (ver la nota de abajo).
    E_P = np.asarray(_d['flux_err_emp' if 'flux_err_emp' in _cols
                        else 'flux_err'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P}
    for _c in ('apcorr', 'npix_eff', 'flags'):
        if _c in _cols:
            EXTRA_P[_c] = np.asarray(_d[_c])
    _h.close()
    try:
        MODO_P = (nb.load_qc('stages/spec_psffit_qc.json', RUN_ID).get('errors') or {}).get('mode')
    except Exception:
        MODO_P = None
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'espectro del compañero · psffit, el método canónico (C4)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c4_psffit'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Figura de paper — el espectro sin binar, con su error y sus líneas

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    from astropy.io import fits
    ROOT_P = nb.project_root()
    METHOD_P = 'psffit_star'
    PRODUCT_P = 'spec_psffit_star.fits'
    TARGET_P = (nb.run_target(RUN_ID) or RUN_ID).replace(' ', '')
    _h = fits.open(nb.run_dir(RUN_ID) / 'stages' / PRODUCT_P)
    _d = _h[1].data
    _cols = list(_d.columns.names)
    # La unidad viaja con el dato (BUNIT); no hay default silencioso.
    BUNIT_P = _h[1].header.get('BUNIT') or 'ADU'
    W_P = np.asarray(_d['wave_A'], float)
    F_P = np.asarray(_d['flux'], float)
    # El empírico manda; `flux_err` es el que eligió la etapa y solo
    # aporta algo cuando NO es el empírico (ver la nota de abajo).
    E_P = np.asarray(_d['flux_err_emp' if 'flux_err_emp' in _cols
                        else 'flux_err'], float)
    E_ALT_P = np.asarray(_d['flux_err'], float)
    EXTRA_P = {'flux_err_stat': E_ALT_P}
    for _c in ('apcorr', 'npix_eff', 'flags'):
        if _c in _cols:
            EXTRA_P[_c] = np.asarray(_d[_c])
    _h.close()
    try:
        MODO_P = (nb.load_qc('stages/spec_psffit_qc.json', RUN_ID).get('errors') or {}).get('mode')
    except Exception:
        MODO_P = None
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'espectro de la PRIMARIA · psffit (C4)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c4_psffit'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper_star' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper_star' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper_star' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Ajuste lineal por canal** `a·P_estrella + b·P_compañero + plano`: toda la no-linealidad se resuelve aguas arriba (C1/B3). Es el método primario de la literatura.
- **Simultáneo estrella+compañero+plano**: maneja el gradiente del halo de frente, sin sobre-sustracción; positivo y físico en el borde → **método canónico**. · [`docs/2026-07-09_d1_canonical_method_decision.md`](../docs/2026-07-09_d1_canonical_method_decision.md)
- Ajuste sano: χ²ᵣ≈1.03, cond 10.7, rho_ab 0.17 (separables), estrella recuperada 1.014, crosstalk 0.03.
- Salvedad: `rho_bc`≈0.45 (1303 canales >0.5), compañero débil parcialmente degenerado con el plano; errores empíricos (M5 rojo).


## Conclusión (registrada)

**C4: método canónico `psffit`; ajuste lineal simultáneo estrella+compañero+plano; χ²ᵣ≈1.03; residual limpio; no-detección.**

- **Fecha:** cadena D1 v2 sobre el run realineado (2026-07-09).
- **Ajuste:** por canal `a·P★ + b·P_c + (c₀+c₁y+c₂x)`; región estrella 20px / compañero 12px; 1705 px/ajuste.
- **Validación:** estrella recuperada (1.014 vs apertura grande), crosstalk 0.03, rho_ab 0.17 (separables).
- **Residual:** halo + compañero removidos (queda el anillo ~4–5% de C1).
- **Salvedad:** rho_bc≈0.45 (degeneración compañero-fondo en canales débiles); errores empíricos.
- **Downstream:** es el canónico de D1/D2/E1/E3; el par primario de D1 es psffit vs optimal_psfsub. El producto final lleva `cont_runmed_biasref` para G3.
